In [237]:
import pandas as pd
import numpy as np
import plotly.express as px
import loaders

In [238]:
book_authors = {
    "Genesis": "Moses",
    "Exodus": "Moses",
    "Leviticus": "Moses",
    "Numbers": "Moses",
    "Deuteronomy": "Moses",
    "Joshua": "Joshua",
    "Judges": "Samuel; Nathan; Gad",
    "Ruth": "Samuel",
    "1 Samuel": "Samuel; Nathan; Gad",
    "2 Samuel": "Gad; Nathan",
    "1 Kings": "Jeremiah",
    "2 Kings": "Jeremiah",
    "1 Chronicles": "Ezra",
    "2 Chronicles": "Ezra",
    "Ezra": "Ezra",
    "Nehemiah": "Nehemiah",
    "Esther": "Mordecai",
    "Job": "Unknown",
    "Psalms": "David and others",
    "Proverbs": "Solomon; Agur; Lemuel",
    "Ecclesiastes": "Solomon",
    "Song of Songs": "Solomon",
    "Isaiah": "Isaiah",
    "Jeremiah": "Jeremiah",
    "Lamentations": "Jeremiah",
    "Ezekiel": "Ezekiel",
    "Daniel": "Daniel",
    "Hosea": "Hosea",
    "Joel": "Joel",
    "Amos": "Amos",
    "Obadiah": "Obadiah",
    "Jonah": "Jonah",
    "Micah": "Micah",
    "Nahum": "Nahum",
    "Habakkuk": "Habakkuk",
    "Zephaniah": "Zephaniah",
    "Haggai": "Haggai",
    "Zechariah": "Zechariah",
    "Malachi": "Malachi",
    "Matthew": "Matthew",
    "Mark": "Mark",
    "Luke": "Luke",
    "John": "John",
    "Acts": "Luke",
    "Romans": "Paul",
    "1 Corinthians": "Paul",
    "2 Corinthians": "Paul",
    "Galatians": "Paul",
    "Ephesians": "Paul",
    "Philippians": "Paul",
    "Colossians": "Paul",
    "1 Thessalonians": "Paul",
    "2 Thessalonians": "Paul",
    "1 Timothy": "Paul",
    "2 Timothy": "Paul",
    "Titus": "Paul",
    "Philemon": "Paul",
    "Hebrews": "Unknown",
    "James": "James",
    "1 Peter": "Peter",
    "2 Peter": "Peter",
    "1 John": "John",
    "2 John": "John",
    "3 John": "John",
    "Jude": "Jude",
    "Revelation": "John"
}

In [239]:
sept = loaders.load_sept()
tisch = loaders.load_tisch()
references = loaders.load_munged_references()

### Does Matthew Contain more OT references than John? 

In [240]:
matthew_v_john = references[(references['n_start'].str[0] == "Matthew") | (references['n_start'].str[0] == "John")]
display(matthew_v_john.head(2))
display(matthew_v_john.tail(2))

,n_start,n_end,o_start,o_end,new_text,old_text
0,"(Matthew, 1, 1)","(Matthew, 1, 2)","(Genesis, 25, 19)","(Genesis, 25, 20)",βίβλος γενέσεως ἰησοῦ χριστοῦ υἱοῦ δαυεὶδ υἱοῦ...,και αυται αι γενεσεις ισαακ του υιου αβρααμ αβ...
1,"(Matthew, 1, 21)","(Matthew, 1, 21)","(Genesis, 16, 11)","(Genesis, 16, 11)",τέξεται δὲ υἱὸν καὶ καλέσεις τὸ ὄνομα αὐτοῦ ἰη...,και ειπεν αυτη ο αγγελος κυριου ιδου συ εν γασ...


,n_start,n_end,o_start,o_end,new_text,old_text
726,"(John, 19, 24)","(John, 19, 24)","(Psalms, 22, 18)","(Psalms, 22, 19)",εἶπαν οὖν πρὸς ἀλλήλους μὴ σχίσωμεν αὐτόν ἀλλὰ...,διεμερισαντο τα ιματια μου εαυτοις και επι τον...
727,"(John, 20, 17)","(John, 20, 17)","(Genesis, 45, 9)","(Genesis, 45, 9)",λέγει αὐτῇ ἰησοῦς μή μου ἅπτου οὔπω γὰρ ἀναβέβ...,σπευσαντες ουν αναβητε προς τον πατερα μου και...


In [241]:
px.histogram(
    x = matthew_v_john['n_start'].str[0],
    labels = {'x':'', 'y':''},
    title = "Which has more Allusions, Matthew or John?"
)

In [242]:
matthew_v_john['n_start'].str[0].value_counts().to_frame()

,count
n_start,
Matthew,192
John,58


#### Percentage of Author Allusions

In [243]:
fig = px.pie(
    references['n_start'].str[0].map(book_authors).value_counts().to_frame().reset_index(),
    names = "n_start",
    values= "count",
    title= "Percentage of references made by each Author in the NT."
)
fig.update_traces(textposition='inside', textinfo='percent+label', showlegend=False)
fig.show()

#### Most quoted old testatement texts

In [244]:
references['o_start'].str[0].map(book_authors)

0          Moses
1          Moses
2          Moses
3          Moses
4          Moses
          ...   
1479        Ezra
1480    Jeremiah
1481       Moses
1482        Ezra
1483    Jeremiah
Name: o_start, Length: 1484, dtype: object

In [245]:
ot_references = references['o_start'].str[0].value_counts().to_frame().reset_index()
rare_index = ot_references[ot_references['count'] < 12].index
ot_references.loc[rare_index, "o_start"] = "Other"
ot_references['color'] = ot_references['o_start'].apply(lambda book: "red" if book == "Other" else 'blue')
ot_references.head()


,o_start,count,color
0,Psalms,398,blue
1,Isaiah,184,blue
2,Deuteronomy,174,blue
3,Genesis,148,blue
4,Exodus,81,blue


In [246]:
fig = px.histogram(
    ot_references,
    x = "o_start",
    y = "count",
    color = "color",
    labels = {'x':'Old Testament Book', 'y':'Number of Allusions'},
    title = "Number of Times each Old Testament Book is Referenced."
)
fig.update_traces(showlegend=False)
fig.show()

#### Make a network.

In [247]:
import networkx as nx

In [248]:
revelations_references = references.query("n_start.str[0] == 'Revelation'").copy()
revelations_references

,n_start,n_end,o_start,o_end,new_text,old_text
1375,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 72, 19)","(Psalms, 72, 19)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,και ευλογητον το ονομα της δοξης αυτου εις τον...
1376,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 83, 17)","(Psalms, 83, 17)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,αισχυνθητωσαν και ταραχθητωσαν εις τον αιωνα τ...
1377,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 119, 44)","(Psalms, 119, 45)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,και φυλαξω τον νομον σου δια παντος εις τον αι...
1378,"(Revelation, 2, 7)","(Revelation, 2, 8)","(Ezekiel, 31, 8)","(Ezekiel, 31, 8)",ὁ ἔχων οὖς ἀκουσάτω τί τὸ πνεῦμα λέγει ταῖς ἐκ...,κυπαρισσοι τοιαυται ουκ εγενηθησαν εν τω παραδ...
1379,"(Revelation, 3, 12)","(Revelation, 3, 12)","(Psalms, 44, 20)","(Psalms, 44, 20)",ὁ νικῶν ποιήσω αὐτὸν στῦλον ἐν τῷ ναῷ τοῦ θεοῦ...,ει επελαθομεθα του ονοματος του θεου ημων και ...
...,...,...,...,...,...,...
1479,"(Revelation, 22, 18)","(Revelation, 22, 19)","(2 Chronicles, 34, 21)","(2 Chronicles, 34, 22)",μαρτυρῶ ἐγὼ παντὶ τῷ ἀκούοντι τοὺς λόγους τῆς ...,πορευθητε ζητησατε τον κυριον περι εμου και πε...
1480,"(Revelation, 22, 18)","(Revelation, 22, 19)","(Jeremiah, 25, 13)","(Jeremiah, 25, 15)",μαρτυρῶ ἐγὼ παντὶ τῷ ἀκούοντι τοὺς λόγους τῆς ...,και επαξω επι την γην εκεινην παντας τους λογο...
1481,"(Revelation, 22, 19)","(Revelation, 22, 20)","(Deuteronomy, 28, 58)","(Deuteronomy, 28, 58)",καὶ ἐάν τις ἀφέλῃ ἀπὸ τῶν λόγων τοῦ βιβλίου τῆ...,εαν μη εισακουσητε ποιειν παντα τα ρηματα του ...
1482,"(Revelation, 22, 19)","(Revelation, 22, 20)","(2 Chronicles, 34, 21)","(2 Chronicles, 34, 22)",καὶ ἐάν τις ἀφέλῃ ἀπὸ τῶν λόγων τοῦ βιβλίου τῆ...,πορευθητε ζητησατε τον κυριον περι εμου και πε...


In [249]:
network = nx.DiGraph()
network.add_node('Revelation', type="NT")
network.add_nodes_from(
    list(set(revelations_references['o_start'].str[0])), type = "OT"
)


In [250]:
import networkx as nx

# Create an empty directed graph
G = nx.DiGraph()

# Add nodes
G.add_node('Revelation', type='NT')
G.add_nodes_from(
    list(set(revelations_references['o_start'].str[0])),
    type = "OT"
)

# Add edges (optionally with weight)
G.add_edge('Revelation', 'Genesis', weight=3)
G.add_edge('Revelation', 'Exodus', weight=2)
G.add_edge('Revelation', 'Daniel', weight=1)



pos = nx.spring_layout(G, seed=42)  # force-directed layout

import plotly.graph_objects as go

# Edges
edge_x, edge_y = [], []
for source, target in G.edges():
    x0, y0 = pos[source]
    x1, y1 = pos[target]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(x=edge_x, y=edge_y,
                        line=dict(width=2, color='gray'),
                        hoverinfo='none',
                        mode='lines')

# Nodes
node_x = [pos[node][0] for node in G.nodes()]
node_y = [pos[node][1] for node in G.nodes()]
node_trace = go.Scatter(x=node_x, y=node_y,
                        mode='markers+text',
                        text=list(G.nodes()),
                        textposition='top center',
                        marker=dict(size=20, color=['red' if G.nodes[n]['type']=='NT' else 'lightblue' for n in G.nodes()]))



fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    xaxis=dict(showgrid=False, showticklabels=False, zeroline=False),
    yaxis=dict(showgrid=False, showticklabels=False, zeroline=False)
)
fig.show()


KeyError: 'type'

In [ ]:
revelations_references

,n_start,n_end,o_start,o_end,new_text,old_text
1375,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 72, 19)","(Psalms, 72, 19)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,και ευλογητον το ονομα της δοξης αυτου εις τον...
1376,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 83, 17)","(Psalms, 83, 17)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,αισχυνθητωσαν και ταραχθητωσαν εις τον αιωνα τ...
1377,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 119, 44)","(Psalms, 119, 45)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,και φυλαξω τον νομον σου δια παντος εις τον αι...
1378,"(Revelation, 2, 7)","(Revelation, 2, 8)","(Ezekiel, 31, 8)","(Ezekiel, 31, 8)",ὁ ἔχων οὖς ἀκουσάτω τί τὸ πνεῦμα λέγει ταῖς ἐκ...,κυπαρισσοι τοιαυται ουκ εγενηθησαν εν τω παραδ...
1379,"(Revelation, 3, 12)","(Revelation, 3, 12)","(Psalms, 44, 20)","(Psalms, 44, 20)",ὁ νικῶν ποιήσω αὐτὸν στῦλον ἐν τῷ ναῷ τοῦ θεοῦ...,ει επελαθομεθα του ονοματος του θεου ημων και ...
...,...,...,...,...,...,...
1479,"(Revelation, 22, 18)","(Revelation, 22, 19)","(2 Chronicles, 34, 21)","(2 Chronicles, 34, 22)",μαρτυρῶ ἐγὼ παντὶ τῷ ἀκούοντι τοὺς λόγους τῆς ...,πορευθητε ζητησατε τον κυριον περι εμου και πε...
1480,"(Revelation, 22, 18)","(Revelation, 22, 19)","(Jeremiah, 25, 13)","(Jeremiah, 25, 15)",μαρτυρῶ ἐγὼ παντὶ τῷ ἀκούοντι τοὺς λόγους τῆς ...,και επαξω επι την γην εκεινην παντας τους λογο...
1481,"(Revelation, 22, 19)","(Revelation, 22, 20)","(Deuteronomy, 28, 58)","(Deuteronomy, 28, 58)",καὶ ἐάν τις ἀφέλῃ ἀπὸ τῶν λόγων τοῦ βιβλίου τῆ...,εαν μη εισακουσητε ποιειν παντα τα ρηματα του ...
1482,"(Revelation, 22, 19)","(Revelation, 22, 20)","(2 Chronicles, 34, 21)","(2 Chronicles, 34, 22)",καὶ ἐάν τις ἀφέλῃ ἀπὸ τῶν λόγων τοῦ βιβλίου τῆ...,πορευθητε ζητησατε τον κυριον περι εμου και πε...


In [ ]:
revelations_references['revelations_loc'] = revelations_references.apply(
    lambda row : (":".join(map(str, row['n_start'])) + "-" + str(row['n_end'][-1])),
    axis = 1,
)
revelations_references['old_testament_loc'] = revelations_references.apply(
    lambda row : (":".join(map(str, row['o_start'])) + "-" + str(row['o_start'][-1])),
    axis = 1,
)

revelations_references.head(3)

,n_start,n_end,o_start,o_end,new_text,old_text,revelations_loc,old_testament_loc
1375,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 72, 19)","(Psalms, 72, 19)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,και ευλογητον το ονομα της δοξης αυτου εις τον...,Revelation:1:18-18,Psalms:72:19-19
1376,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 83, 17)","(Psalms, 83, 17)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,αισχυνθητωσαν και ταραχθητωσαν εις τον αιωνα τ...,Revelation:1:18-18,Psalms:83:17-17
1377,"(Revelation, 1, 18)","(Revelation, 1, 18)","(Psalms, 119, 44)","(Psalms, 119, 45)",καὶ ὁ ζῶν καὶ ἐγενόμην νεκρὸς καὶ ἰδοὺ ζῶν εἰμ...,και φυλαξω τον νομον σου δια παντος εις τον αι...,Revelation:1:18-18,Psalms:119:44-44


In [ ]:
len(list(set(revelations_references['old_testament_loc'])))

58

In [ ]:
old_testament_books = revelations_references['o_start'].str[0].value_counts()
old_testament_books['Psalms']

32

In [ ]:
G = nx.DiGraph()
G.add_node(
    "Revelation",
    type="NT"
)

old_testament_books = (revelations_references['o_start'].str[0]).value_counts()
G.add_nodes_from(
    old_testament_books.keys(),
    type = "OT"
)

for book in old_testament_books.items():
    G.add_edge(
        "Revelation",
        book[0],
        weight = book[1]
    )

In [274]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go

# --- 1. Build the graph ---
G = nx.DiGraph()
G.add_node("Revelation", type="NT")

old_testament_books = dict((revelations_references['o_start'].str[0]).value_counts())

G.add_nodes_from(old_testament_books.keys(), type="OT")

for book, count in old_testament_books.items():
    G.add_edge("Revelation", book, weight=count)

# --- 2. Define layout (Revelation center, others around it) ---
center_node = "Revelation"
connected_nodes = list(G.neighbors(center_node))
pos = {center_node: (0, 0)}
radius = 1.8
n = len(connected_nodes)

for i, node in enumerate(connected_nodes):
    angle = 2 * np.pi * i / n
    pos[node] = (radius * np.cos(angle), radius * np.sin(angle))

# --- 3. Node sizes based on edge weights ---
weights = nx.get_edge_attributes(G, "weight")
max_w = max(weights.values()) if weights else 1
node_sizes = {
    node: 55 if node == center_node else 15 + 45 * (weights.get((center_node, node), 0) / max_w)
    for node in G.nodes
}

# --- 4. Build edge trace ---
edge_x, edge_y, edge_text = [], [], []
for src, dst, data in G.edges(data=True):
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    edge_text.append(f"{src} → {dst}: {data['weight']}")

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=1, color="#888"),
    hoverinfo="none",
    mode="lines"
)

# --- 5. Build node trace ---
node_x = [pos[node][0] for node in G.nodes]
node_y = [pos[node][1] for node in G.nodes]
node_text = [node for node in G.nodes]
node_size_list = [1.25 * node_sizes[node] for node in G.nodes]

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=node_text,
    textposition=[
        "middle center" if node == "Revelation" else "middle right"
        for node in G.nodes
    ],
    marker=dict(
        size=node_size_list,
        color=[
            "#f1c40f" if G.nodes[n]["type"] == "NT" else "#3498db"
            for n in G.nodes
        ],
        line=dict(width=2, color="#fff"),
        opacity=1,
    ),
    hovertext=[
        f"{node}: {weights.get((center_node, node), 0)} references" if node != center_node else node
        for node in G.nodes
    ],
    hoverinfo="text"
)

# --- 6. Plot ---
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title=dict(
        text="All the References in Revelations",
        x=0.5,          # Center the title
        y=0.95,         # Raise it slightly above the plot
        xanchor='center',
        yanchor='top',
        font=dict(size=20)
    ),
    showlegend=False,
    margin=dict(l=40, r=40, t=80, b=40),  # add top margin for title
    plot_bgcolor="white",
    hovermode="closest",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)
fig.show()
